# 15 · Model comparison (validation 2023)

Every model with saved validation forecasts, on the same stocks and days. Intervals are 95% block-bootstrap
intervals over trading days (blocks of 20 days, 300 resamples), which respect the correlation between stocks on the
same day. Test-period results are produced once, by the freeze task 16.9.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.analysis.plots import (BLUE_CMAP, HORIZON_COLORS, MODEL_COLORS, TEXT_2, apply_style, bar_style,
                                dot_compare, figures_dir, save)
from src.config import Config
from src.evaluation.metrics import calibration_table, evaluate, join_labels
from src.models.data import label_frame, load_panel
from src.models.walkforward import run_or_load

cfg = Config.load()
apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
val_start, val_end = cfg.period("validation")

In [ ]:
from src.evaluation.bootstrap import metrics_with_intervals
from src.models.walkforward import load_forecasts

panel = load_panel(cfg)
labels = label_frame(panel)
fc = load_forecasts(cfg, "validation")
joined = join_labels(fc, labels)
table = metrics_with_intervals(joined, metrics=("accuracy", "brier", "vol_mae"), n=300, block=20,
                               seed=cfg["models"]["seed"])
table.to_csv(figures_dir(cfg) / "validation_metrics.csv", index=False)
print("models:", ", ".join(sorted(fc.model.unique())))
table.round(4)

In [ ]:
direction_models = [m for m in fc.model.unique() if m != "hist_vol"]
vol_models = [m for m in fc.model.unique() if fc.loc[fc.model == m, "vol_forecast"].notna().any()]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dot_compare(axes[0], table, "accuracy", direction_models, panel.horizons, "accuracy_low", "accuracy_high",
            reference=0.5)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Direction accuracy (95% interval)")
dot_compare(axes[1], table, "brier", direction_models, panel.horizons, "brier_low", "brier_high", fmt="{:.3f}")
axes[1].set_title("Brier score (lower is better)")
fig.tight_layout()
save(fig, cfg, "15_direction_comparison")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
dot_compare(ax, table, "vol_mae", vol_models, panel.horizons, "vol_mae_low", "vol_mae_high", fmt="{:.4f}")
ax.set_title("Volatility forecast error, MAE (95% interval; lower is better)")
save(fig, cfg, "15_volatility_comparison")
plt.show()

## Calibration at 5 days

In [ ]:
cal = calibration_table(joined[joined.horizon_days == 5], bins=10)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for m in [m for m in ["tft", "lstm", "arima", "ma_crossover"] if m in cal.model.unique()]:
    g = cal[cal.model == m]
    ax.plot(g.p_mean, g.up_rate, marker="o", color=MODEL_COLORS[m], label=m)
lims = [0.3, 0.8]
ax.plot(lims, lims, color=TEXT_2, linewidth=1, linestyle="--")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("mean forecast p_up (decile)")
ax.set_ylabel("actual share of 5-day rises")
ax.set_title("Calibration, 5-day horizon")
ax.legend()
save(fig, cfg, "15_calibration_5d")
plt.show()

## By market regime

High VIX: India VIX above its 2016–2022 median at the forecast. Uptrend: the Nifty above its 200-day EMA.

In [ ]:
train_vix = panel.df.loc[panel.df.trade_date <= pd.Timestamp(cfg.period("train")[1]), "vix"].median()
regime = panel.df[["symbol", "trade_date", "vix", "nifty_ema200_gap"]].assign(
    vix_regime=lambda d: np.where(d.vix > train_vix, "high VIX", "low VIX"),
    trend=lambda d: np.where(d.nifty_ema200_gap > 0, "uptrend", "downtrend"))
jr = joined.merge(regime, on=["symbol", "trade_date"])
by_regime = (jr[jr.model.isin(direction_models)]
             .assign(correct=lambda d: ((d.direction > 0) == (d.up == 1)).astype(float))
             .groupby(["horizon_days", "vix_regime", "model"]).correct.mean().unstack("model"))
days = jr.drop_duplicates("trade_date")
print(f"VIX threshold (train median): {train_vix:.3f}; validation days with high VIX: {(days.vix > train_vix).mean():.1%}, "
      f"in a Nifty downtrend: {(days.nifty_ema200_gap <= 0).mean():.1%}")
by_trend = (jr[jr.model.isin(direction_models)]
            .assign(correct=lambda d: ((d.direction > 0) == (d.up == 1)).astype(float))
            .groupby(["horizon_days", "trend", "model"]).correct.mean().unstack("model"))
pd.concat({"by VIX": by_regime, "by trend": by_trend}).round(3)

## Findings

1. **No price-only model beats the constant base rate on direction in 2023.** Base rate 53.1% / 56.2% / 60.6%; LSTM identical (it learned the base rate); ARIMA 47.8% / 55.1% / 58.6%; MA crossover 49.7% / 53.3% / 54.4%; TFT 51.4% / 45.7% / 42.6%.
2. **Intervals are wide at 5 and 10 days** (base rate 50.5%-62.5% at 5 days, 52.6%-69.6% at 10 days): overlapping windows and stocks moving together leave few independent outcomes in one year. The TFT's intervals (40.3%-51.1% at 5 days, 33.8%-50.6% at 10 days) sit mostly below the base rate's.
3. **Volatility:** the LSTM is the only model better than historical volatility: clearly at 1 day (MAE 0.0069, interval 0.0065-0.0074, vs 0.0088, 0.0083-0.0094) and by 10-11% at 5 and 10 days, where the intervals overlap. ARIMA and the TFT are worse than historical volatility.
4. **2023 tests one regime:** India VIX was above its 2016-2022 median on 1.6% of days and the Nifty below its 200-day EMA on 9.3%, so the regime splits rest on few days.
5. **What this sets for the project:** price history alone gives a useful volatility forecast (LSTM) but no direction edge, so any direction skill of the agent system has to come from event evidence. Before freezing the baselines (16.9): tune and calibrate the LSTM and TFT on validation, then run the test period once with --freeze.